# Fine-tuning the pretrained encoder on SST-2 (sentiment)

Take the pretrained backbone (`ckpt2/last.pt`, the UltraFineWeb-refined encoder — falls back to
`ckpt/last.pt`), **discard the MLM head**, and attach a small classification head. We **full
fine-tune** (backbone + head together) — the standard, best-performing BERT recipe.

**No `[CLS]` token:** the model never saw one (OLMo has none; we dropped NSP), so instead of
pooling a `[CLS]` position we **mean-pool the encoder's hidden states** over the real tokens —
often better than `[CLS]` anyway (Sentence-BERT). We pool the *hidden representations* `H`
(shape B×T×768), **not** the MLM token predictions.

In [ ]:
import os, sys, torch
import torch.nn as nn
from torch.utils.data import DataLoader
ROOT = os.path.abspath(".."); sys.path.insert(0, ROOT)
from mini_enc_transformer import Encoder
from mini_enc_transformer import build_tokenizer
from datasets import load_from_disk
from types import SimpleNamespace
device = "cuda" if torch.cuda.is_available() else "cpu"; device

## Config + checkpoint (must match the pretraining architecture)

In [ ]:
cfg = SimpleNamespace(d_model=768, d_embed=128, n_heads=4, n_layers=4, d_k=64, d_v=64)
CKPT = os.path.join(ROOT, "ckpt2", "last.pt")             # UltraFineWeb-refined backbone
if not os.path.exists(CKPT): CKPT = os.path.join(ROOT, "ckpt", "last.pt")  # fallback: phase-1
tk, ids = build_tokenizer("allenai/OLMo-1B-hf")
print("using checkpoint:", CKPT)

## Data — SST-2 (train on `train`, evaluate on `validation`)
The official test labels are hidden (−1), so we report **validation accuracy**, the GLUE-standard
practice. Data was fetched once to `datasets/sst2`.

In [ ]:
ds = load_from_disk(os.path.join(ROOT, "datasets", "sst2"))
train_ds, val_ds = ds["train"], ds["validation"]
print({k: len(v) for k,v in ds.items()})
print(train_ds[0])   # {'sentence': 'hide new secretions ...', 'label': 0, 'idx': 0}

## Tokenize + padded collate (mean-pool needs a real-token mask)

In [ ]:
MAXLEN = 64   # SST-2 sentences are short
def encode(b): return tk(b["sentence"], truncation=True, max_length=MAXLEN)
train_tok = train_ds.map(encode, batched=True)
val_tok   = val_ds.map(encode,   batched=True)

def collate(rows):
    seqs = [r["input_ids"] for r in rows]; L = max(len(s) for s in seqs)
    x = torch.full((len(rows), L), ids["pad_id"], dtype=torch.long)
    m = torch.zeros((len(rows), L), dtype=torch.bool)          # True = real token
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s); m[i, :len(s)] = True
    y = torch.tensor([r["label"] for r in rows])
    return x, m, y

## Model — encoder + mean-pool + linear head

`Encoder(ids)` returns hidden states `H` (B,T,768). We mean-pool over real tokens → one 768-d
vector per sentence → linear → 2 logits. We load only the `encoder.*` weights from the pretrained
`BertForMaskedLM` checkpoint (the `mlm_head.*` weights are dropped).

In [ ]:
class EncoderForSentiment(nn.Module):
    def __init__(self, ckpt_path, cfg, ids, n_classes=2):
        super().__init__()
        self.encoder = Encoder(ids["vocab_size"], cfg.d_model, cfg.d_k, cfg.d_v,
                               cfg.n_heads, cfg.n_layers, pad_id=ids["pad_id"], d_embed=cfg.d_embed)
        sd = torch.load(ckpt_path, map_location="cpu")["model"]
        enc = {k[len("encoder."):]: v for k, v in sd.items() if k.startswith("encoder.")}
        self.encoder.load_state_dict(enc)          # exact: only the encoder subtree
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(cfg.d_model, n_classes)
    def forward(self, x, m):
        H = self.encoder(x)                        # (B,T,768) hidden states (NOT token logits)
        w = m.unsqueeze(-1).float()
        pooled = (H * w).sum(1) / w.sum(1).clamp(min=1)   # mean over real tokens (no [CLS])
        return self.head(self.drop(pooled))

model = EncoderForSentiment(CKPT, cfg, ids).to(device)
sum(p.numel() for p in model.parameters())/1e6

## Full fine-tune (backbone + head), eval on validation

In [ ]:
train_loader = DataLoader(train_tok, batch_size=32, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_tok,   batch_size=64, shuffle=False, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
lossf = nn.CrossEntropyLoss()

@torch.no_grad()
def accuracy():
    model.eval(); c = t = 0
    for x, m, y in val_loader:
        x, m, y = x.to(device), m.to(device), y.to(device)
        c += (model(x, m).argmax(-1) == y).sum().item(); t += y.numel()
    model.train(); return c / t

for ep in range(3):
    for x, m, y in train_loader:
        x, m, y = x.to(device), m.to(device), y.to(device)
        loss = lossf(model(x, m), y)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {ep+1}: val_acc = {accuracy():.4f}")

## Yardstick
BERT-base gets ~**93%** on SST-2 (110M params, 3.3B tokens). We're 28.7M params on a few-hundred-M
tokens — so the honest target is **competitive-at-a-fraction-of-the-size** (~mid-80s%), an
efficiency win, not beating BERT-base outright. To A/B pooling, swap mean-pool for an added
`[CLS]` token (prepend, resize embedding by one row) and compare.